# Студия образовательных программ: воспроизводимый сценарий

Этот ноутбук показывает решение по тем же шагам, которые реализованы в веб-студии и Python/FastAPI backend. Его можно пройти сверху вниз и отдельно проверить материалы кейса, формальные правила и API.

## Как работает решение

1. Методист передаёт запрос, DOCX/PDF и ссылки на источники.
2. Агент извлекает только явно заданные параметры. Если не хватает типа программы, точных часов, аудитории или аттестации, он задаёт до пяти вопросов и останавливается.
3. Cloud.ru Evolution предлагает каркас: бриф, компетенции, модули и учебный план.
4. Python-код проверяет форму программы, целые академические часы, обязательные поля и связи.
5. Методист исправляет и утверждает каркас. Агент не может утвердить его от имени человека.
6. После утверждения по каждому модулю создаются тесты, практические задания и лекции.
7. Аудит проверяет полноту комплекта и связь каждого результата обучения с тестом и практическим заданием.
8. Курс, история и версии сохраняются через SQLAlchemy; результат можно выгрузить в DOCX, PPTX, HTML и JSON.

Модель отвечает за содержание, обычный код — за проверяемые правила, методист — за источники и итоговое решение.


## Шаг 1. Проверяем входные материалы

В `materials/` лежат два примера программ АЦР и пять открытых запросов с чек-листами. Ниже DOCX читается стандартной библиотекой Python — без отправки документов во внешний сервис.


In [1]:
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

ROOT = Path("..").resolve()
MATERIALS = ROOT / "materials"

def docx_text(path: Path) -> str:
    with zipfile.ZipFile(path) as archive:
        xml = archive.read("word/document.xml")
    root = ET.fromstring(xml)
    lines = []
    for paragraph in root.iter("{http://schemas.openxmlformats.org/wordprocessingml/2006/main}p"):
        text = "".join(node.text or "" for node in paragraph.iter("{http://schemas.openxmlformats.org/wordprocessingml/2006/main}t"))
        if text.strip():
            lines.append(text.strip())
    return "\n".join(lines)

files = {
    "Общеразвивающая, 24 ч": MATERIALS / "program-general-24h.docx",
    "Повышение квалификации, 248 ч": MATERIALS / "program-qualification-248h.docx",
    "Пять открытых запросов": MATERIALS / "open-cases.docx",
}
for title, path in files.items():
    text = docx_text(path)
    print(f"{title}: {len(text)} знаков, файл на месте — {path.name}")


Общеразвивающая, 24 ч: 29256 знаков, файл на месте — program-general-24h.docx
Повышение квалификации, 248 ч: 27211 знаков, файл на месте — program-qualification-248h.docx
Пять открытых запросов: 9916 знаков, файл на месте — open-cases.docx


## Шаг 2. Отделяем генерацию от формальных проверок

Модель не должна проверять собственную арифметику. Для каждого модуля код сравнивает его объём с суммой лекций, практик, лабораторных и самостоятельной работы. Затем сумма модулей и аттестации сравнивается с объёмом программы.

В примере стажировки разные строки документа дают разные распределения 246 часов. Поэтому дополнительно сравниваем строки между собой и не переносим противоречие в новую программу.


In [2]:
def check_hours(modules, attestation, duration):
    notes = []
    for module in modules:
        kinds = sum(module[key] for key in ("lecture", "practice", "lab", "self_study"))
        if kinds != module["hours"]:
            notes.append(f"{module['title']}: по видам работ {kinds} ч, в модуле {module['hours']} ч")
    total = sum(module["hours"] for module in modules) + attestation
    if total != duration:
        notes.append(f"Итог {total} ч, запрошено {duration} ч")
    return {"passed": not notes, "notes": notes}

sample_24 = [
    {"title": "Модуль 1", "hours": 5, "lecture": 2, "practice": 2, "lab": 0, "self_study": 1},
    {"title": "Модуль 2", "hours": 6, "lecture": 2, "practice": 2, "lab": 0, "self_study": 2},
    {"title": "Модуль 3", "hours": 6, "lecture": 2, "practice": 3, "lab": 0, "self_study": 1},
    {"title": "Модуль 4", "hours": 7, "lecture": 2, "practice": 3, "lab": 0, "self_study": 2},
]
print("Пример 24 ч:", check_hours(sample_24, attestation=0, duration=24))

header_distribution = {"lecture": 0, "practice": 0, "lab": 0, "self_study": 246}
total_row_distribution = {"lecture": 0, "practice": 66, "lab": 6, "self_study": 174}
print("Строки образца 248 ч совпадают:", header_distribution == total_row_distribution)
if header_distribution != total_row_distribution:
    print("Несоответствие найдено — распределение должен подтвердить методист.")


Пример 24 ч: {'passed': True, 'notes': []}
Строки образца 248 ч совпадают: False
Несоответствие найдено — распределение должен подтвердить методист.


## Шаг 3. Фиксируем проверяемые ожидания по пяти запросам

Каждый открытый запрос проверяет отдельное поведение агента. Эти условия используются как приёмочные чек-листы; скрытые запросы устроены аналогично, но содержат другие темы, аудитории и объёмы.


In [3]:
CASES = [
    {"id": 1, "kind": "general", "hours": 16, "title": "Python для школьников", "must": "практики не меньше 8 ч, тест на 10 вопросов, слайды занятий 1 и 2"},
    {"id": 2, "kind": "qualification", "hours": 36, "title": "ИИ в работе учителя", "must": "удостоверение, профстандарт «Педагог», только сервисы, доступные в России"},
    {"id": 3, "kind": "qualification", "hours": 72, "title": "Анализ данных на Python", "must": "pandas, визуализация, SQL, статистика и вводный Python; слайды модуля про pandas"},
    {"id": 4, "kind": None, "hours": None, "title": "Кибербезопасность, запрос неполный", "must": "сначала не больше пяти вопросов, потом 72 ч повышения квалификации"},
    {"id": 5, "kind": "qualification", "hours": 256, "title": "Стажировка DevOps", "must": "254 ч самостоятельной работы, 2 ч аттестации, лекции и практики нули, ошибку 248 ч не копировать"},
]
for case in CASES:
    print(f"{case['id']}. {case['title']} — {case['must']}")


1. Python для школьников — практики не меньше 8 ч, тест на 10 вопросов, слайды занятий 1 и 2
2. ИИ в работе учителя — удостоверение, профстандарт «Педагог», только сервисы, доступные в России
3. Анализ данных на Python — pandas, визуализация, SQL, статистика и вводный Python; слайды модуля про pandas
4. Кибербезопасность, запрос неполный — сначала не больше пяти вопросов, потом 72 ч повышения квалификации
5. Стажировка DevOps — 254 ч самостоятельной работы, 2 ч аттестации, лекции и практики нули, ошибку 248 ч не копировать


## Шаг 4. Проверяем реализованный Python API

Перед выполнением следующей ячейки запустите backend из корня проекта:

```bash
python -m pip install -r backend/requirements.txt
npm run backend:migrate
npm run backend
```

Проверка `/health` показывает, что сервис доступен. Затем подготовленный курс отправляется в `/api/audit`: этот маршрут не обращается к модели и детерминированно проверяет каркас, часы, комплект и источники.


In [4]:
import json
import urllib.request

API = "http://127.0.0.1:8000"

try:
    with urllib.request.urlopen(f"{API}/health", timeout=10) as response:
        print("Backend:", json.load(response))

    course = json.loads((ROOT / "fixtures" / "course-contract.json").read_text(encoding="utf-8"))
    request = urllib.request.Request(
        f"{API}/api/audit",
        data=json.dumps(course).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=20) as response:
        audit = json.load(response)

    print("Итог аудита:", audit["summary"])
    for item in audit["items"]:
        print(f"[{item['status'].upper()}] {item['label']}")
except Exception as error:
    print("Python backend недоступен:", error)
    print("Запустите `npm run backend` в корне репозитория.")


Backend:

 {'status': 'ok', 'service': 'edu-course-python'}
Итог аудита: {'pass': 4, 'warning': 0, 'fail': 2}
[PASS] Выбран шаблон программы
[PASS] Есть название и аудитория
[PASS] Часы сходятся: 16 из 16
[FAIL] Базовый синтаксис: лекция, задание и тест
[FAIL] Результат «Пишет программу с ветвлением» проверяется двумя способами
[PASS] Источники и референсы указаны


## Шаг 5. Запускаем полный сценарий с Cloud.ru Evolution

`POST /api/run` проходит тот же цикл, что и студия: уточнение данных → каркас → проверка → утверждение → комплект → итоговый аудит. Для запроса к модели нужен `CLOUDRU_API_KEY` в корневом `.env`.

Автоутверждение используется только в техническом прогоне пяти кейсов. В интерфейсе каркас утверждает методист.

In [5]:
RUN_MODEL = False  # Поставьте True только для живого запроса к Cloud.ru

if RUN_MODEL:
    case = CASES[0]
    payload = {
        "request": (
            "Общеразвивающая программа «Python для школьников», 16 академических часов, "
            "очно. Аудитория — школьники 12–15 лет. Итоговая аттестация — зачёт."
        ),
        "references": docx_text(files["Пять открытых запросов"]),
        "caseId": case["id"],
        "autoApprove": True,
    }
    request = urllib.request.Request(
        f"{API}/api/run",
        data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=300) as response:
        result = json.load(response)
    print("Название:", result["course"]["brief"]["title"])
    print("Отчёт:", result["report"])
else:
    print("Живой прогон отключён. Установите RUN_MODEL = True после запуска backend и проверки ключа.")

Живой прогон отключён. Установите RUN_MODEL = True после запуска backend и проверки ключа.


## Шаг 6. Открываем сохранённый курс и печатаем планы

Строка в «Моих курсах» делает то же самое: `GET /api/courses`, затем `GET /api/courses/{id}`. Дальше печатаются компетенции, план лекций, семинаров и тестов.

Если сервер не ответил или курса ещё нет в базе, те же шаги идут локально без модели: каркас, проверка часов, утверждение методиста, комплект.

In [ ]:
import json
import sys
import urllib.request

sys.path.insert(0, str(ROOT))
API = "http://127.0.0.1:8000"
ANALYST = (
    "Общеразвивающая программа «ИИ для аналитиков: от данных к решению». "
    "24 академических часа, очно. Аудитория — бизнес- и дата-аналитики с Excel и SQL, "
    "без опыта обучения моделей. Итоговая аттестация — зачёт."
)

def module_title(course, module_id):
    return next((item["title"] for item in course["modules"] if item["id"] == module_id), module_id)

def print_course(course):
    brief = course["brief"]
    print(f"{brief['title']} · {brief['audience']} · {brief['durationHours']} ч")
    print("Компетенции:")
    for node in course["competencies"]["nodes"]:
        print(f"  {node['id']}. {node['name']}")
        if node.get("knowledge"):
            print("    знания:", "; ".join(node["knowledge"]))
        if node.get("skills"):
            print("    умения:", "; ".join(node["skills"]))
        if node.get("abilities"):
            print("    навыки:", "; ".join(node["abilities"]))
    print("Лекции:")
    for lecture in course["lectures"]:
        print(f"  {module_title(course, lecture['moduleId'])}, {lecture['durationMin']} мин")
        for index, item in enumerate(lecture.get("outline") or [], start=1):
            print(f"    {index}. {item}")
    print("Семинары:")
    for seminar in course["seminars"]:
        print(f"  {module_title(course, seminar['moduleId'])}")
        for task in seminar["tasks"]:
            print(f"    {task['title']} · {task['timeMin']} мин")
            for step in task["steps"]:
                print(f"      — {step}")
    print("Тесты:")
    for assessment in course["assessments"]:
        print(f"  {module_title(course, assessment['moduleId'])}: {len(assessment['questions'])} вопр.")
        for index, question in enumerate(assessment["questions"], start=1):
            print(f"    {index}. {question['stem']}")

opened = None
try:
    with urllib.request.urlopen(f"{API}/api/courses", timeout=10) as response:
        listing = json.load(response)["courses"]
    match = next((item for item in listing if "аналитик" in str(item.get("title") or "").lower()), None)
    if match:
        with urllib.request.urlopen(f"{API}/api/courses/{match['id']}", timeout=10) as response:
            opened = json.load(response)["course"]
        print("Открыт сохранённый курс", match["id"])
    else:
        print("В базе нет курса про аналитиков. Собираю его локально.")
except Exception as error:
    print("Список курсов недоступен:", error)
    print("Собираю программу локально, без модели.")

if opened is None:
    from backend.app.agent import approve_framework, assemble_local_course, fill_missing_kit
    from backend.app.domain import framework_items

    course = assemble_local_course([ANALYST])
    print("Каркас:", [(item["id"], item["ok"]) for item in framework_items(course)])
    course = approve_framework(course)
    course = fill_missing_kit(course, None, [ANALYST])
    opened = course.as_api()

print_course(opened)


## Шаг 7. Что получает методист

После проверки каркаса интерфейс позволяет:

- утвердить или вернуть каркас на доработку;
- собрать тесты с ключами, задания с критериями и лекции;
- изменить отдельный раздел без пересборки всего курса;
- открыть детерминированный отчёт по часам, полноте, связям и источникам;
- сохранить очередную версию курса в SQLite локально или PostgreSQL при развёртывании;
- выгрузить программу и материалы в DOCX, PPTX, HTML и JSON.

Границы текущей реализации: точное визуальное совпадение DOCX с бланком АЦР ещё нужно подтвердить; LMS-экспорт, pgvector и объектное хранилище остаются следующими этапами.